# 🛵 Rappi Intelligent Analysis System
### Prueba Técnica — AI Engineer

---

**Stack:** Python · Streamlit · Groq API (Llama 3.3 70b) · Pandas · Plotly

Este notebook cubre la estructura completa del sistema, con código real y resultados, siguiendo la estructura de la presentación:

1. Contexto y approach
2. Demo: 5 preguntas de diferente complejidad
3. Insights automáticos
4. Decisiones técnicas
5. Limitaciones y próximos pasos

---
## Parte 0 — Setup

Importamos todo lo necesario. Como los módulos del proyecto están en la misma carpeta, basta con agregar el path.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from dotenv import load_dotenv
load_dotenv()  # carga GROQ_API_KEY desde .env

import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook'

print('Setup OK')

---
## Parte 1 — Contexto y Approach (3 min)

### ¿Cómo interpreté el problema?

El desafío era construir un **chatbot analítico** que pudiera responder preguntas en lenguaje natural sobre métricas operacionales de Rappi (12.573 KPIs por zona y semana en 9 países), y generar un **reporte automático de insights** sin que el usuario tenga que hacer consultas.

### Decisión de arquitectura clave: separar LLM de cálculo

```
Usuario pregunta en español
        ↓
bot.py → LLM (Groq/Llama) detecta INTENT + extrae PARÁMETROS → JSON
        ↓
analytics.py → función pandas ejecuta el cálculo REAL
        ↓
bot.py → LLM redacta respuesta en español con el resultado como contexto
        ↓
app.py → muestra respuesta + gráfico automático según intent
```

**¿Por qué esta separación?**
- Los LLMs alucinan cuando calculan sobre datos reales → Pandas garantiza exactitud
- El LLM hace lo que hace bien: entender lenguaje natural y redactar
- Pandas hace lo que hace bien: agrupar, filtrar, ordenar miles de filas

### Módulos del sistema

| Módulo | Responsabilidad |
|---|---|
| `data_loader.py` | Carga Excel, limpieza, expone DataFrames wide y long |
| `analytics.py` | Motor analítico puro — todos los cálculos con pandas |
| `bot.py` | Orquestador — intent parsing, memoria conversacional, LLM |
| `insights.py` | Escaneo automático: anomalías, tendencias, correlaciones |
| `charts.py` | Gráficos Plotly según tipo de consulta |
| `app.py` | Interfaz Streamlit — chat + reporte de insights |

---
## Parte 2 — Los Datos

### data_loader.py — Carga y limpieza

El archivo Excel tiene 3 sheets. Los cargamos y exponemos en **dos formatos**:
- **Wide** (una fila por zona+métrica): ideal para rankings y valores puntuales
- **Long** (una fila por zona+métrica+semana): ideal para tendencias temporales

In [ ]:
from data_loader import load_data, get_data_summary

df_metrics, df_orders, df_metrics_long, df_orders_long = load_data()

summary = get_data_summary()
print('=== RESUMEN DEL DATASET ===')
print(f"Métricas (wide):  {summary['metrics_rows']:>6,} filas  |  {summary['metrics_unique_zones']} zonas  |  {summary['metrics_unique_countries']} países  |  {summary['metrics_unique_metrics']} métricas")
print(f"Órdenes  (wide):  {summary['orders_rows']:>6,} filas  |  {summary['orders_unique_zones']} zonas")
print(f"Métricas (long):  {summary['metrics_long_rows']:>6,} filas")
print(f"Órdenes  (long):  {summary['orders_long_rows']:>6,} filas")
print(f"\nNulos en métricas: {summary['metrics_null_pct']}%  (solo semanas antiguas — zonas nuevas)")
print(f"Nulos en órdenes:  {summary['orders_null_pct']}%  (zonas sin datos de volumen)")

In [ ]:
# Formato WIDE — una fila por zona+métrica
# Columnas L8W_ROLL...L0W_ROLL son los valores semanales
print('Formato WIDE (df_metrics):')
df_metrics.head(3)

In [ ]:
# Formato LONG — una fila por zona+métrica+semana
# Este formato es ideal para graficar tendencias temporales
print('Formato LONG (df_metrics_long):')
df_metrics_long.head(6)

In [ ]:
# Países disponibles
print('Países:', ', '.join(summary['metrics_countries']))
print('\nMétricas:')
for m in summary['metrics_list']:
    print(f'  • {m}')

---
## Parte 3 — Demo del Bot: 5 Preguntas

Para cada pregunta muestro:
1. La pregunta en lenguaje natural
2. El intent + parámetros que detecta el LLM
3. La función de analytics que se ejecuta
4. El resultado
5. El gráfico generado automáticamente

> **Nota:** en la demo en vivo, todo esto ocurre en segundos dentro del chat de Streamlit.

### Pregunta 1 — Simple: Top N zonas por métrica

**Pregunta:** *"¿Cuáles son las 5 zonas con mayor Lead Penetration en Colombia?"*

**Intent detectado por el LLM:**
```json
{"intent": "top_k", "params": {"metric": "Lead Penetration", "n": 5, "country": "CO", "week": "L0W_ROLL"}}
```

In [ ]:
import analytics as an

# Función que ejecuta analytics.py
resultado = an.get_top_zones(
    metric='Lead Penetration',
    n=5,
    country='CO',
    week='L0W_ROLL'
)
resultado

In [ ]:
# Gráfico automático generado por charts.py
from charts import chart_top_k

fig = chart_top_k(metric='Lead Penetration', n=5, country='CO')
fig.show()

### Pregunta 2 — Comparación entre grupos

**Pregunta:** *"Compara el desempeño de Perfect Orders entre zonas Wealthy y Non Wealthy en México"*

**Intent detectado:**
```json
{"intent": "compare", "params": {"metric": "Perfect Orders", "group_col": "ZONE_TYPE", "filters": {"COUNTRY": "MX"}}}
```

In [ ]:
resultado = an.compare_groups(
    metric='Perfect Orders',
    group_col='ZONE_TYPE',
    filters={'COUNTRY': 'MX'}
)
print('Comparación Wealthy vs Non Wealthy — Perfect Orders en MX:')
resultado

In [ ]:
from charts import chart_compare

fig = chart_compare(
    metric='Perfect Orders',
    group_col='ZONE_TYPE',
    filters={'COUNTRY': 'MX'}
)
fig.show()

### Pregunta 3 — Tendencia temporal (evolución semana a semana)

**Pregunta:** *"Muestra la evolución de Gross Profit UE en las últimas 8 semanas para las zonas top de Brasil"*

Primero encontramos una zona top de Brasil, luego graficamos su tendencia.

**Intent detectado:**
```json
{"intent": "trend", "params": {"zone": "<zona>", "metric": "Gross Profit UE", "n_weeks": 8}}
```

In [ ]:
# Primero encontramos la zona #1 de Brasil en Gross Profit UE
top_br = an.get_top_zones(metric='Gross Profit UE', n=1, country='BR')
zona_br = top_br.iloc[0]['ZONE']
print(f'Zona top de Brasil en Gross Profit UE: {zona_br}')

# Ahora obtenemos su tendencia de 8 semanas
tendencia = an.get_trend(zone=zona_br, metric='Gross Profit UE', n_weeks=8)
print('\nTendencia semana a semana (incluye delta y cambio %):')
tendencia

In [ ]:
from charts import chart_trend

fig = chart_trend(zone=zona_br, metric='Gross Profit UE', n_weeks=8)
fig.show()

### Pregunta 4 — Análisis multivariable (combinación de métricas)

**Pregunta:** *"¿Qué zonas tienen alto Lead Penetration pero bajo Perfect Orders? Son zonas que adquieren usuarios pero no los retienen."*

**Intent detectado:**
```json
{"intent": "multivariable_scan", "params": {"metric_high": "Lead Penetration", "metric_low": "Perfect Orders", "threshold": 0.7}}
```

**¿Cómo funciona?** Calcula percentiles por país → zonas en el top 30% de LP Y el bottom 30% de PO.

In [ ]:
resultado = an.multivariable_scan(
    metric_high='Lead Penetration',
    metric_low='Perfect Orders',
    threshold=0.7   # top/bottom 30%
)
print(f'Zonas con alto Lead Penetration y bajo Perfect Orders: {len(resultado)}')
resultado.head(10)

In [ ]:
from charts import chart_multivariable

fig = chart_multivariable(
    metric_high='Lead Penetration',
    metric_low='Perfect Orders',
    threshold=0.7
)
fig.show()

### Pregunta 5 — Compleja: Zonas problemáticas (combina múltiples análisis)

**Pregunta:** *"¿Cuáles son las zonas más problemáticas en Argentina?"*

**Intent detectado:**
```json
{"intent": "problematic_zones", "params": {"country": "AR", "threshold": 0.10, "min_weeks": 3}}
```

**¿Qué hace?** Combina dos detecciones:
1. **Anomalías** → caídas >10% entre la semana pasada y la actual
2. **Tendencias declinantes** → deterioro 3+ semanas consecutivas

In [ ]:
# Detección 1: Anomalías semana a semana (caídas >10%)
anomalias = an.detect_anomalies(threshold=0.10)
anomalias_ar = anomalias[anomalias['COUNTRY'] == 'AR']

print(f'Anomalías en Argentina: {len(anomalias_ar)} (de {len(anomalias)} totales)')
print('\nPeores caídas esta semana:')
anomalias_ar[anomalias_ar['direction'] == 'caida'].head(8)

In [ ]:
# Detección 2: Tendencias declinantes 3+ semanas consecutivas
declinantes = an.detect_declining_trends(min_weeks=3)
declinantes_ar = declinantes[declinantes['COUNTRY'] == 'AR'] if not declinantes.empty else declinantes

print(f'Zonas con deterioro continuo en Argentina: {len(declinantes_ar)}')
if not declinantes_ar.empty:
    print('\nTendencias más preocupantes:')
    display(declinantes_ar.head(8))
else:
    print('No se detectaron tendencias declinantes en Argentina.')

In [ ]:
# El bot combina ambos análisis y los presenta como un dict unificado
# Esto es exactamente lo que bot.py le pasa al LLM para que redacte la respuesta
resultado_problematicas = {
    'total_anomalias': len(anomalias_ar),
    'total_declinantes': len(declinantes_ar),
    'anomalias_semana': anomalias_ar[anomalias_ar['direction']=='caida'].head(5).to_dict(orient='records'),
    'tendencias_declinantes': declinantes_ar.head(5).to_dict(orient='records') if not declinantes_ar.empty else []
}

import json
print('Contexto que recibe el LLM para redactar la respuesta:')
print(json.dumps({k: v if not isinstance(v, list) else f'[{len(v)} registros]' for k, v in resultado_problematicas.items()}, indent=2))

### Follow-up — Memoria conversacional

Una vez que el usuario hace una pregunta, el sistema recuerda el contexto. Si pregunta:

**Pregunta:** *"¿Y en Colombia?"*

El bot detecta `intent: follow_up` y hereda todo el contexto anterior, cambiando solo `country: CO`.

```python
# last_context después de la pregunta anterior:
last_context = {"intent": "problematic_zones", "country": "AR", "threshold": 0.10, "min_weeks": 3}

# LLM detecta: {"intent": "follow_up", "params": {"country": "CO"}}

# bot.py hace merge:
merged = {"intent": "problematic_zones", "country": "CO", "threshold": 0.10, "min_weeks": 3}
#                                                    ↑ solo esto cambió
```

---
## Parte 4 — Insights Automáticos (5 min)

`insights.py` escanea todos los datos automáticamente (sin que el usuario pregunte nada) y genera un reporte ejecutivo en Markdown con:
1. Resumen ejecutivo con KPIs
2. Anomalías (cambios >10% semana a semana)
3. Tendencias preocupantes (deterioro 3+ semanas)
4. Correlaciones entre métricas
5. Oportunidades (zonas con mayor crecimiento de órdenes)

In [ ]:
# Corremos el análisis de anomalías para ver los números del resumen ejecutivo
anomalias_all = an.detect_anomalies(threshold=0.10)
declinantes_all = an.detect_declining_trends(min_weeks=3)
correlaciones = an.detect_correlations(min_corr=0.6)
crecimiento = an.growth_analysis(n_weeks=5)

n_caidas  = len(anomalias_all[anomalias_all['direction'] == 'caida'])  if not anomalias_all.empty else 0
n_mejoras = len(anomalias_all[anomalias_all['direction'] == 'mejora']) if not anomalias_all.empty else 0

print('=== RESUMEN EJECUTIVO ===')
print(f'Zonas con caída  >10% esta semana:           {n_caidas}')
print(f'Zonas con mejora >10% esta semana:           {n_mejoras}')
print(f'Zona+métricas en deterioro 3+ semanas:       {len(declinantes_all)}')
print(f'Pares de métricas con correlación fuerte:    {len(correlaciones)}')
if not anomalias_all.empty:
    top_metric = anomalias_all[anomalias_all['direction']=='caida']['METRIC'].value_counts().index[0]
    print(f'Métrica con más caídas esta semana:          {top_metric}')
if not crecimiento.empty:
    print(f'Zona con mayor crecimiento de órdenes (5sem): {crecimiento.iloc[0]["ZONE"]} ({crecimiento.iloc[0]["COUNTRY"]})')

In [ ]:
# Top caídas más fuertes
print('Top 10 caídas más fuertes esta semana (L1W → L0W):')
if not anomalias_all.empty:
    caidas = anomalias_all[anomalias_all['direction']=='caida'].head(10)
    display(caidas[['COUNTRY','CITY','ZONE','METRIC','L1W_ROLL','L0W_ROLL','change_pct']].rename(
        columns={'change_pct': 'cambio_%', 'L1W_ROLL': 'sem_ant', 'L0W_ROLL': 'sem_act'}
    ))

In [ ]:
# Correlaciones detectadas entre métricas
print('Pares de métricas correlacionadas (r ≥ 0.6):')
if not correlaciones.empty:
    display(correlaciones)
else:
    print('No se detectaron correlaciones fuertes con el umbral actual.')

In [ ]:
# Zonas con mayor crecimiento de órdenes
print('Top 10 zonas con mayor crecimiento de órdenes (últimas 5 semanas):')
if not crecimiento.empty:
    display(crecimiento.head(10))

# Gráfico
from charts import chart_growth
fig = chart_growth(n_weeks=5, top_n=15)
if fig:
    fig.show()

In [ ]:
# Generar el reporte completo en Markdown
from insights import generate_report
from IPython.display import Markdown

reporte = generate_report()
Markdown(reporte)

---
## Parte 5 — Decisiones Técnicas (5 min)

### 1. Elección del LLM: Groq + Llama 3.3 70b

| Opción | Pro | Con |
|---|---|---|
| **Groq + Llama 3.3 70b** ✅ | Gratis, ultra rápido (~200 tok/s), JSON mode, 70b capaz | Rate limits en free tier (6k TPM) |
| OpenAI GPT-4o | Más capaz en edge cases | Costo, más lento |
| LLM local (Ollama) | Sin rate limits, privado | Hardware intensivo, lento |

Para una prueba técnica, Groq free tier es el mejor trade-off: calidad suficiente + 0 costo + latencia real.

### 2. Separación LLM vs Pandas (arquitectura de 2 llamadas)

Cada turno del usuario genera **2 llamadas al LLM**:
- **Llamada 1** (intent detection, temp=0.0): parse estricto → JSON. Recibe solo el mensaje actual + contexto anterior (no el historial completo, para ahorrar tokens)
- **Llamada 2** (response generation, temp=0.4): redacta en español con el resultado de Pandas como contexto. Recibe el historial completo para continuidad narrativa

### 3. Memoria conversacional con last_context

En lugar de pasar el historial completo al parser de intents (caro en tokens), guardamos un dict `last_context` con el intent y parámetros del último turno. Esto permite resolver follow-ups (`"¿y en México?"`) con solo los parámetros que cambian.

In [ ]:
# Ejemplo concreto de cómo funciona el merge de follow-up en bot.py

# Estado después de: "Top 5 zonas de Lead Penetration en Colombia"
last_context = {
    'intent': 'top_k',
    'metric': 'Lead Penetration',
    'n': 5,
    'country': 'CO',
    'week': 'L0W_ROLL'
}

# El LLM detecta que "¿y en México?" es un follow_up con solo country cambiando
followup_params = {'country': 'MX'}

# bot.py hace este merge:
merged = {k: v for k, v in last_context.items() if k != 'intent'}
merged.update(followup_params)
intent = last_context['intent']  # hereda el intent anterior

print(f'Intent: {intent}')
print(f'Params merged: {merged}')
print(f'\n→ Ejecuta: get_top_zones(metric="Lead Penetration", n=5, country="MX", week="L0W_ROLL")')

### 4. Intents soportados

| Intent | Ejemplo | Función analytics |
|---|---|---|
| `top_k` | ¿Las 5 mejores zonas en Lead Penetration? | `get_top_zones()` |
| `compare` | ¿Wealthy vs Non Wealthy en MX? | `compare_groups()` |
| `trend` | Evolución de Chapinero las últimas 8 semanas | `get_trend()` |
| `aggregate` | Promedio de Lead Penetration por país | `aggregate_metric()` |
| `multivariable_scan` | Alto LP pero bajo PO | `multivariable_scan()` |
| `growth_inference` | Zonas que más crecen en órdenes | `growth_analysis()` |
| `benchmark` | Zonas divergentes en Colombia | `benchmark_analysis()` |
| `problematic_zones` | ¿Zonas problemáticas en AR? | `detect_anomalies()` + `detect_declining_trends()` |
| `correlations` | ¿Qué métricas están correlacionadas? | `detect_correlations()` |
| `follow_up` | ¿Y en México? / ¿Y las últimas 3 semanas? | *hereda intent anterior* |
| `conversational` | ¿De qué hablamos antes? | *solo historial, sin analytics* |

### 5. Benchmark analysis (detección de outliers por z-score)

In [ ]:
# Zonas con desempeño divergente respecto a sus pares del mismo país
# Usa z-score > 1.5 por métrica dentro del grupo COUNTRY + ZONE_TYPE
benchmark = an.benchmark_analysis(country='CO', zone_type='Wealthy')
print(f'Zonas Wealthy en Colombia con performance divergente: {len(benchmark)}')
if not benchmark.empty:
    display(benchmark.head(10))

---
## Parte 6 — Limitaciones y Próximos Pasos (2 min)

### Limitaciones actuales

| Limitación | Detalle |
|---|---|
| **Rate limits de Groq free tier** | 6,000 TPM → si hay muchos usuarios simultáneos, hay espera |
| **Sin autenticación** | Cualquiera con la URL puede usar la app |
| **Intent detection brittle** | Nombres de métricas con typos pueden no mapearse (aunque el LLM intenta corregirlos) |
| **Sin persistencia** | El historial de conversación vive en `st.session_state` → se pierde al recargar |
| **Solo datos estáticos** | El sistema no conecta a una DB en tiempo real; depende del Excel |

### Con más tiempo implementaría

1. **Streaming de respuestas** — Groq soporta `stream=True`, mejoraría mucho la UX mostrando el texto mientras se genera
2. **Cache semántico** — guardar pares (pregunta, resultado) para no recalcular consultas repetidas
3. **Conectar a una fuente de datos real** — reemplazar el Excel por una conexión a BigQuery/Snowflake con datos actualizados
4. **Evaluación de intents** — dataset de pares (pregunta → intent esperado) para medir la tasa de accuracy del parser
5. **Exportar reporte a PDF** — el reporte de insights en Markdown es fácil de convertir con `weasyprint`
6. **Soporte multilingüe** — el sistema prompt está optimizado para español, pero podría adaptarse

---
## Apéndice — Arquitectura del Intent Parser (bot.py)

Esta es la lógica central que orquesta todo:

In [ ]:
# Pseudocódigo simplificado del flujo en bot.chat()

def chat_simplified(user_message, messages, last_context):
    """
    Función principal — orquesta todo el flujo.
    """
    
    # PASO 1 — LLM detecta intent (temperatura 0, JSON mode)
    intent_result = llm_call(
        system=INTENT_PROMPT,    # prompt con esquemas JSON de todos los intents
        user=user_message,
        context_hint=last_context  # solo el último intent, no todo el historial
    )
    intent = intent_result['intent']   # 'top_k', 'trend', 'follow_up', ...
    params = intent_result['params']
    
    # PASO 2 — Si es follow_up, mergear con contexto anterior
    if intent == 'follow_up':
        intent = last_context['intent']   # hereda el intent previo
        params = {**last_context, **params}  # pisa solo lo que cambió
    
    # PASO 3 — Si es conversacional, responder solo del historial
    if intent == 'conversational':
        return llm_call(system=RESPONSE_PROMPT, messages=messages, user=user_message)
    
    # PASO 4 — Ejecutar analytics (PANDAS, no el LLM)
    analytics_result = execute_analytics(intent, params)
    
    # PASO 5 — LLM redacta la respuesta en español
    response = llm_call(
        system=RESPONSE_PROMPT,
        messages=messages,          # historial completo para continuidad
        user=f'Pregunta: {user_message}\n\nResultado:\n{analytics_result}'
    )
    
    return response, {'intent': intent, **params}

print('Arquitectura correcta — ver bot.py:437 para el código real')

---

## Para correr la demo en vivo

```bash
# 1. Asegurarse de tener el .env con GROQ_API_KEY
# 2. Instalar dependencias
pip install -r requirements.txt

# 3. Lanzar la app
streamlit run app.py
# → http://localhost:8501
```

**Preguntas sugeridas para la demo en vivo:**
1. `¿Cuáles son las 5 zonas con mayor Lead Penetration en Colombia?`
2. `Compara Perfect Orders entre Wealthy y Non Wealthy en México`
3. *(follow-up)* `¿Y en Brasil?`
4. `¿Qué zonas tienen alto Lead Penetration pero bajo Perfect Orders?`
5. `¿Cuáles son las zonas problemáticas en Argentina?`
6. *(ir a tab Insights)* Hacer clic en "Generar reporte"

---
*Repositorio: https://github.com/valehepla/rappi-chatbot*